In [1]:
from reload import reload_package
reload_package(namespace=globals())

Reloaded 0 modules, rebound 0 names.


In [2]:
import altair as alt
import numpy as np
import polars as pl
import torch

from ts_uncertainty.forecasting import ForecastModel
from ts_uncertainty.metrics import mean_absolute_scaled_error, scaled_quantile_loss

alt.renderers.enable("jupyter", offline=True)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Data

In [3]:
from ts_uncertainty.chaotic import ChaoticGenerator
from ts_uncertainty.injection import Injector
from ts_uncertainty.plotting import plot_sample, plot_forecasts
from ts_uncertainty.synthetic import SyntheticGenerator
from ts_uncertainty.real import RealSampler

# data settings
context_length = 256
horizon_length = 128
samples_batch = 100
seed = 42

/home/richard/thesis-ts-uncertainty/.venv/lib/python3.12/site-packages/dysts/base.py:22: UserWarning: Numba not installed. Falling back to no JIT compilation.
  warnings.warn("Numba not installed. Falling back to no JIT compilation.")


## Synthetic Samples

In [4]:
sin_gen = SyntheticGenerator(
    context_length=context_length,
    horizon=horizon_length,
    seasonal=True,
    covariate=False,
    periods=(24,),             # period of the sine
    # n_periods=1,
    n_harmonics=3,             # 1 harmonic = pure sine, no extra shape
    # level_range=(0.0, 0.0),    # centered at 0
    slope_range=(0.0, 0.0),    # no trend
    # amplitude_range=(1.0, 1.0),
    seed=seed
)

# generate samples
samples_synthetic = sin_gen.sample_batch(samples_batch)

In [5]:
sample_to_plot = 0
plot_sample(samples_synthetic[sample_to_plot], title=f"Synthetic Sample {sample_to_plot}")

## Chaotic Samples

In [6]:
chaos_gen = ChaoticGenerator(
    context_length=context_length,
    horizon=horizon_length,
    systems=None,                    # None = all ~120 default systems
    dim=0,                           # observed coordinate, None = random per sample
    pts_per_period_range=(20, 30),   # time steps per dominant period (smaller = more irregular)
    level_range=(0.0, 0.0),          # centered at 0
    slope_range=(0.0, 0.0),          # no trend
    amplitude_range=(1.0, 1.0),      # z-normalised signal, std ≈ 1
    seed=seed,
)

# generate samples
samples_chaotic = chaos_gen.sample_batch(samples_batch)

In [7]:
sample_to_plot = 0
plot_sample(samples_chaotic[sample_to_plot], title=f"Chaotic Sample {sample_to_plot}")

## Real Sample
Download the data from the notebook [data_download.ipynb](data_download.ipynb) before.

In [8]:
data_path_test = "../data/test.csv"
data_test = pl.read_csv(data_path_test, try_parse_dates=True)

number_series = len(data_test['series_index'].unique())
print(f"Number of time series in the test set: {number_series}")
data_test.head()

Number of time series in the test set: 361


timestamp,value,series_name,series_index
datetime[μs],f64,str,i64
2025-11-19 10:00:00,0.36,"""home_electricity""",0
2025-11-19 10:15:00,0.124,"""home_electricity""",0
2025-11-19 10:30:00,0.108,"""home_electricity""",0
2025-11-19 10:45:00,0.088,"""home_electricity""",0
2025-11-19 11:00:00,0.116,"""home_electricity""",0


In [9]:
real_gen = RealSampler(
    df=data_test,
    context_length=context_length,
    horizon=horizon_length,
    normalize='none', # dont normalize
    seed=seed,
)

# generate samples
samples_real = real_gen.sample_batch(samples_batch)

In [10]:
sample_to_plot = 0
plot_sample(samples_real[sample_to_plot], title=f"Real Sample {sample_to_plot}")

# Models

In [11]:
models = {
    "AutoTheta": {"name": "autotheta"},
    "Chronos-2": {'name': 'chronos2'},
    "TiRex-2": {'name': 'tirex2'},
    "Timestral (123M)": {'name': 'timestral', 'size': '123M'},
}

# prepare and download all models
for key, m in models.items():
    print(f"Preparing model: {key}")
    model = ForecastModel(**m)

Preparing model: AutoTheta
Preparing model: Chronos-2


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Preparing model: TiRex-2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Preparing model: Timestral (123M)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

# Experiments

In [98]:
def evaluate_models_on_samples(models, samples, clean_samples=None):
    """clean_samples: the same samples before injection (same order); their context gives the
    MASE / SQL scale, so an injection in the context doesn't change the denominator."""
    results = []
    scores_mase = []
    scores_sql = []

    for key, m in models.items():
        print(f"Evaluating model: {key}")
        model = ForecastModel(**m)
        fc = model.forecast(samples)

        fc_m = fc.metric_inputs()
        if clean_samples is None:
            y_scale = fc_m["y_train"]  # (T, B)
        else:
            y_scale = np.stack([s.context for s in clean_samples], axis=1)  # (T, B)

        score_mase = mean_absolute_scaled_error(fc_m["y_true"], fc_m["y_pred"], y_scale, fc_m["season_length"])
        score_sql = scaled_quantile_loss(fc_m["y_true"], fc_m["y_quantiles"], y_scale, fc_m["quantiles"], fc_m["season_length"])

        results.append(fc)
        scores_mase.append(score_mase.mean())
        scores_sql.append(score_sql.mean())

    scores = pl.DataFrame({
        "model": list(models.keys()),
        "mase": [float(x.squeeze()) for x in scores_mase],
        "sql": [float(x.squeeze()) for x in scores_sql],
    })
    return results, scores


def summarize_scores(scores: dict) -> pl.DataFrame:
    """Long score table, one row per (dataset, model).

    scores: either one run {dataset: DataFrame}, giving columns mase, sql, ...;
    or several runs {run: {dataset: DataFrame}}, giving columns mase_<run>, sql_<run>, ...
    """
    def to_long(fc_scores: dict[str, pl.DataFrame]) -> pl.DataFrame:
        long = pl.concat([df.with_columns(pl.lit(name).alias("dataset")) for name, df in fc_scores.items()])
        return long.select("dataset", "model", pl.exclude("dataset", "model"))

    if all(isinstance(v, pl.DataFrame) for v in scores.values()):  # single run
        return to_long(scores)

    tables = []
    for run, fc_scores in scores.items():
        long = to_long(fc_scores)
        tables.append(long.select("dataset", "model", *[pl.col(m).alias(f"{m}_{run}") for m in long.columns[2:]]))

    out = tables[0]
    for t in tables[1:]:
        out = out.join(t, on=["dataset", "model"], how="full", coalesce=True, maintain_order="left")

    # group columns by metric: mase_clean, mase_noise_h, ..., sql_clean, sql_noise_h, ...
    runs = list(scores)
    metric_cols = sorted(out.columns[2:], key=lambda c: (c.split("_", 1)[0], runs.index(c.split("_", 1)[1])))
    return out.select("dataset", "model", *metric_cols)


def delta_scores(summary: pl.DataFrame, reference: str = "clean") -> pl.DataFrame:
    """Mean degradation vs. the reference run: <metric>_<run> - <metric>_<reference>, one row per (dataset, model)."""
    metrics = sorted({c.rsplit("_", 1)[0] for c in summary.columns if c.endswith(f"_{reference}")})
    deltas = [
        (pl.col(c) - pl.col(f"{m}_{reference}")).alias(f"d_{c}")
        for m in metrics
        for c in summary.columns
        if c.startswith(f"{m}_") and c != f"{m}_{reference}"
    ]
    return summary.select("dataset", "model", *deltas)


In [99]:
batch_size = 100  # define the batch size for evaluation
data_samples = {
    "synthetic": samples_synthetic[:batch_size],
    "chaotic": samples_chaotic[:batch_size],
    "real": samples_real[:batch_size],
}

## No Injection

In [100]:
fc_results = {}
fc_scores = {}
for type_s, s in data_samples.items():
    print(f"Forecasting {type_s} samples..., number of samples: {len(s)}")
    results, scores = evaluate_models_on_samples(models, s)
    # save results and scores for a dataset
    fc_results[type_s] = results
    fc_scores[type_s] = scores

Forecasting synthetic samples..., number of samples: 100
Evaluating model: AutoTheta


Evaluating model: Chronos-2


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Evaluating model: TiRex-2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

... (15 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: Timestral (123M)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [101]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=3):
    print(summarize_scores(fc_scores))

shape: (12, 4)
┌───────────┬──────────────────┬───────┬───────┐
│ dataset   ┆ model            ┆ mase  ┆ sql   │
│ ---       ┆ ---              ┆ ---   ┆ ---   │
│ str       ┆ str              ┆ f64   ┆ f64   │
╞═══════════╪══════════════════╪═══════╪═══════╡
│ synthetic ┆ AutoTheta        ┆ 0.000 ┆ 3.014 │
│ synthetic ┆ Chronos-2        ┆ 0.055 ┆ 0.045 │
│ synthetic ┆ TiRex-2          ┆ 0.128 ┆ 0.129 │
│ synthetic ┆ Timestral (123M) ┆ 0.034 ┆ 0.028 │
│ chaotic   ┆ AutoTheta        ┆ 5.331 ┆ 4.844 │
│ chaotic   ┆ Chronos-2        ┆ 3.192 ┆ 2.639 │
│ chaotic   ┆ TiRex-2          ┆ 3.116 ┆ 2.467 │
│ chaotic   ┆ Timestral (123M) ┆ 2.926 ┆ 2.302 │
│ real      ┆ AutoTheta        ┆ 2.021 ┆ 2.349 │
│ real      ┆ Chronos-2        ┆ 1.129 ┆ 0.911 │
│ real      ┆ TiRex-2          ┆ 1.230 ┆ 0.986 │
│ real      ┆ Timestral (123M) ┆ 1.316 ┆ 0.999 │
└───────────┴──────────────────┴───────┴───────┘


In [102]:
dataset_to_plot = "synthetic"  # choose from "synthetic", "chaotic", "real"
sample_to_plot = 5

forecasts = dict(zip(models.keys(), fc_results[dataset_to_plot]))
plot_forecasts(
    data_samples[dataset_to_plot][sample_to_plot],
    forecasts,
    index=sample_to_plot,
    exclude_from_y=[list(models)[0]],  # AutoTheta: its wide bands would squash the plot
)

/tmp/ipykernel_2215/1661976593.py:5: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  plot_forecasts(


## Noise

### Horizon only

In [104]:
inj = Injector(
    noise=True,
    spikes=False,
    shift=False,
    shift_covariate=False,
    spike_covariate=False,
    noise_region="horizon",
    seed=seed,
)

# inject anomalies into the samples
data_samples_injected = {}
for type_s, s in data_samples.items():
    print(f"Injecting anomalies into {type_s} samples..., number of samples: {len(s)}")
    data_samples_injected[type_s] = [inj(x, relative=type_s == "real") for x in s]

fc_results_noise_h = {}
fc_scores_noise_h = {}
for type_s, s in data_samples_injected.items():
    print(f"Forecasting {type_s} samples..., number of samples: {len(s)}")
    results, scores = evaluate_models_on_samples(models, s, clean_samples=data_samples[type_s])
    # save results and scores for a dataset
    fc_results_noise_h[type_s] = results
    fc_scores_noise_h[type_s] = scores

Injecting anomalies into synthetic samples..., number of samples: 100
Injecting anomalies into chaotic samples..., number of samples: 100
Injecting anomalies into real samples..., number of samples: 100
Forecasting synthetic samples..., number of samples: 100
Evaluating model: AutoTheta


Evaluating model: Chronos-2


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Evaluating model: TiRex-2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

... (15 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: Timestral (123M)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [105]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=3):
    print(summarize_scores(fc_scores_noise_h))

shape: (12, 4)
┌───────────┬──────────────────┬───────┬───────┐
│ dataset   ┆ model            ┆ mase  ┆ sql   │
│ ---       ┆ ---              ┆ ---   ┆ ---   │
│ str       ┆ str              ┆ f64   ┆ f64   │
╞═══════════╪══════════════════╪═══════╪═══════╡
│ synthetic ┆ AutoTheta        ┆ 0.772 ┆ 3.149 │
│ synthetic ┆ Chronos-2        ┆ 0.776 ┆ 0.737 │
│ synthetic ┆ TiRex-2          ┆ 0.790 ┆ 0.683 │
│ synthetic ┆ Timestral (123M) ┆ 0.774 ┆ 0.751 │
│ chaotic   ┆ AutoTheta        ┆ 5.382 ┆ 4.876 │
│ chaotic   ┆ Chronos-2        ┆ 3.324 ┆ 2.754 │
│ chaotic   ┆ TiRex-2          ┆ 3.231 ┆ 2.562 │
│ chaotic   ┆ Timestral (123M) ┆ 3.059 ┆ 2.418 │
│ real      ┆ AutoTheta        ┆ 2.134 ┆ 2.389 │
│ real      ┆ Chronos-2        ┆ 1.311 ┆ 1.092 │
│ real      ┆ TiRex-2          ┆ 1.399 ┆ 1.138 │
│ real      ┆ Timestral (123M) ┆ 1.500 ┆ 1.167 │
└───────────┴──────────────────┴───────┴───────┘


In [106]:
dataset_to_plot = "synthetic"  # choose from "synthetic", "chaotic", "real"
sample_to_plot = 5

forecasts = dict(zip(models.keys(), fc_results_noise_h[dataset_to_plot]))
plot_forecasts(
    data_samples_injected[dataset_to_plot][sample_to_plot],
    forecasts,
    index=sample_to_plot,
    exclude_from_y=[list(models)[0]],  # AutoTheta: its wide bands would squash the plot
)

/tmp/ipykernel_2215/2482385561.py:5: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  plot_forecasts(


### Context only

In [107]:
inj = Injector(
    noise=True,
    spikes=False,
    shift=False,
    shift_covariate=False,
    spike_covariate=False,
    noise_region="context",
    seed=seed,
)

# inject anomalies into the samples
data_samples_injected = {}
for type_s, s in data_samples.items():
    print(f"Injecting anomalies into {type_s} samples..., number of samples: {len(s)}")
    data_samples_injected[type_s] = [inj(x, relative=type_s == "real") for x in s]

fc_results_noise_c = {}
fc_scores_noise_c = {}
for type_s, s in data_samples_injected.items():
    print(f"Forecasting {type_s} samples..., number of samples: {len(s)}")
    results, scores = evaluate_models_on_samples(models, s, clean_samples=data_samples[type_s])
    # save results and scores for a dataset
    fc_results_noise_c[type_s] = results
    fc_scores_noise_c[type_s] = scores

Injecting anomalies into synthetic samples..., number of samples: 100
Injecting anomalies into chaotic samples..., number of samples: 100
Injecting anomalies into real samples..., number of samples: 100
Forecasting synthetic samples..., number of samples: 100
Evaluating model: AutoTheta


Evaluating model: Chronos-2


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Evaluating model: TiRex-2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

... (15 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: Timestral (123M)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [108]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=3):
    print(summarize_scores(fc_scores_noise_c))

shape: (12, 4)
┌───────────┬──────────────────┬───────┬───────┐
│ dataset   ┆ model            ┆ mase  ┆ sql   │
│ ---       ┆ ---              ┆ ---   ┆ ---   │
│ str       ┆ str              ┆ f64   ┆ f64   │
╞═══════════╪══════════════════╪═══════╪═══════╡
│ synthetic ┆ AutoTheta        ┆ 0.324 ┆ 1.092 │
│ synthetic ┆ Chronos-2        ┆ 0.205 ┆ 0.271 │
│ synthetic ┆ TiRex-2          ┆ 0.312 ┆ 0.360 │
│ synthetic ┆ Timestral (123M) ┆ 0.224 ┆ 0.298 │
│ chaotic   ┆ AutoTheta        ┆ 5.287 ┆ 5.126 │
│ chaotic   ┆ Chronos-2        ┆ 3.273 ┆ 2.652 │
│ chaotic   ┆ TiRex-2          ┆ 3.255 ┆ 2.587 │
│ chaotic   ┆ Timestral (123M) ┆ 3.273 ┆ 2.577 │
│ real      ┆ AutoTheta        ┆ 2.087 ┆ 2.280 │
│ real      ┆ Chronos-2        ┆ 1.461 ┆ 1.166 │
│ real      ┆ TiRex-2          ┆ 1.449 ┆ 1.160 │
│ real      ┆ Timestral (123M) ┆ 1.325 ┆ 1.074 │
└───────────┴──────────────────┴───────┴───────┘


In [109]:
dataset_to_plot = "synthetic"  # choose from "synthetic", "chaotic", "real"
sample_to_plot = 5

forecasts = dict(zip(models.keys(), fc_results_noise_c[dataset_to_plot]))
plot_forecasts(
    data_samples_injected[dataset_to_plot][sample_to_plot],
    forecasts,
    index=sample_to_plot,
    exclude_from_y=[list(models)[0]],  # AutoTheta: its wide bands would squash the plot
)

/tmp/ipykernel_2215/4188322343.py:5: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  plot_forecasts(


### Both

In [110]:
inj = Injector(
    noise=True,
    spikes=False,
    shift=False,
    shift_covariate=False,
    spike_covariate=False,
    noise_region="both",
    seed=seed,
)

# inject anomalies into the samples
data_samples_injected = {}
for type_s, s in data_samples.items():
    print(f"Injecting anomalies into {type_s} samples..., number of samples: {len(s)}")
    data_samples_injected[type_s] = [inj(x, relative=type_s == "real") for x in s]

fc_results_noise_b = {}
fc_scores_noise_b = {}
for type_s, s in data_samples_injected.items():
    print(f"Forecasting {type_s} samples..., number of samples: {len(s)}")
    results, scores = evaluate_models_on_samples(models, s, clean_samples=data_samples[type_s])
    # save results and scores for a dataset
    fc_results_noise_b[type_s] = results
    fc_scores_noise_b[type_s] = scores

Injecting anomalies into synthetic samples..., number of samples: 100
Injecting anomalies into chaotic samples..., number of samples: 100
Injecting anomalies into real samples..., number of samples: 100
Forecasting synthetic samples..., number of samples: 100
Evaluating model: AutoTheta


Evaluating model: Chronos-2


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Evaluating model: TiRex-2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

... (15 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: Timestral (123M)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [111]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=3):
    print(summarize_scores(fc_scores_noise_b))

shape: (12, 4)
┌───────────┬──────────────────┬───────┬───────┐
│ dataset   ┆ model            ┆ mase  ┆ sql   │
│ ---       ┆ ---              ┆ ---   ┆ ---   │
│ str       ┆ str              ┆ f64   ┆ f64   │
╞═══════════╪══════════════════╪═══════╪═══════╡
│ synthetic ┆ AutoTheta        ┆ 0.838 ┆ 1.216 │
│ synthetic ┆ Chronos-2        ┆ 0.802 ┆ 0.622 │
│ synthetic ┆ TiRex-2          ┆ 0.839 ┆ 0.656 │
│ synthetic ┆ Timestral (123M) ┆ 0.807 ┆ 0.626 │
│ chaotic   ┆ AutoTheta        ┆ 5.342 ┆ 5.155 │
│ chaotic   ┆ Chronos-2        ┆ 3.395 ┆ 2.746 │
│ chaotic   ┆ TiRex-2          ┆ 3.359 ┆ 2.665 │
│ chaotic   ┆ Timestral (123M) ┆ 3.391 ┆ 2.664 │
│ real      ┆ AutoTheta        ┆ 2.150 ┆ 2.301 │
│ real      ┆ Chronos-2        ┆ 1.621 ┆ 1.268 │
│ real      ┆ TiRex-2          ┆ 1.598 ┆ 1.246 │
│ real      ┆ Timestral (123M) ┆ 1.483 ┆ 1.166 │
└───────────┴──────────────────┴───────┴───────┘


In [112]:
dataset_to_plot = "synthetic"  # choose from "synthetic", "chaotic", "real"
sample_to_plot = 5

forecasts = dict(zip(models.keys(), fc_results_noise_b[dataset_to_plot]))
plot_forecasts(
    data_samples_injected[dataset_to_plot][sample_to_plot],
    forecasts,
    index=sample_to_plot,
    exclude_from_y=[list(models)[0]],  # AutoTheta: its wide bands would squash the plot
)

/tmp/ipykernel_2215/2017913783.py:5: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  plot_forecasts(


### Summary

In [113]:
summary = summarize_scores({
    "clean": fc_scores,
    "noise_h": fc_scores_noise_h,
    "noise_c": fc_scores_noise_c,
    "noise_b": fc_scores_noise_b,
})
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=3, tbl_width_chars=1000):
    print(summary)
    print(delta_scores(summary))

shape: (12, 10)
┌───────────┬──────────────────┬────────────┬──────────────┬──────────────┬──────────────┬───────────┬─────────────┬─────────────┬─────────────┐
│ dataset   ┆ model            ┆ mase_clean ┆ mase_noise_h ┆ mase_noise_c ┆ mase_noise_b ┆ sql_clean ┆ sql_noise_h ┆ sql_noise_c ┆ sql_noise_b │
│ ---       ┆ ---              ┆ ---        ┆ ---          ┆ ---          ┆ ---          ┆ ---       ┆ ---         ┆ ---         ┆ ---         │
│ str       ┆ str              ┆ f64        ┆ f64          ┆ f64          ┆ f64          ┆ f64       ┆ f64         ┆ f64         ┆ f64         │
╞═══════════╪══════════════════╪════════════╪══════════════╪══════════════╪══════════════╪═══════════╪═════════════╪═════════════╪═════════════╡
│ synthetic ┆ AutoTheta        ┆ 0.000      ┆ 0.772        ┆ 0.324        ┆ 0.838        ┆ 3.014     ┆ 3.149       ┆ 1.092       ┆ 1.216       │
│ synthetic ┆ Chronos-2        ┆ 0.055      ┆ 0.776        ┆ 0.205        ┆ 0.802        ┆ 0.045     ┆ 0.737      

## Spikes

### Without covariates

#### Horizon only

In [114]:
inj = Injector(
    noise=False,
    spikes=True,
    shift=False,
    shift_covariate=False,
    spike_covariate=False,
    spike_region="horizon",
    spike_prob=0.04,
    seed=seed,
)

# inject anomalies into the samples
data_samples_injected = {}
for type_s, s in data_samples.items():
    print(f"Injecting anomalies into {type_s} samples..., number of samples: {len(s)}")
    data_samples_injected[type_s] = [inj(x, relative=type_s == "real") for x in s]

fc_results_spike_h = {}
fc_scores_spike_h = {}
for type_s, s in data_samples_injected.items():
    print(f"Forecasting {type_s} samples..., number of samples: {len(s)}")
    results, scores = evaluate_models_on_samples(models, s, clean_samples=data_samples[type_s])
    # save results and scores for a dataset
    fc_results_spike_h[type_s] = results
    fc_scores_spike_h[type_s] = scores

Injecting anomalies into synthetic samples..., number of samples: 100
Injecting anomalies into chaotic samples..., number of samples: 100
Injecting anomalies into real samples..., number of samples: 100
Forecasting synthetic samples..., number of samples: 100
Evaluating model: AutoTheta


Evaluating model: Chronos-2


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Evaluating model: TiRex-2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

... (15 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: Timestral (123M)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [115]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=3):
    print(summarize_scores(fc_scores_spike_h))

shape: (12, 4)
┌───────────┬──────────────────┬───────┬───────┐
│ dataset   ┆ model            ┆ mase  ┆ sql   │
│ ---       ┆ ---              ┆ ---   ┆ ---   │
│ str       ┆ str              ┆ f64   ┆ f64   │
╞═══════════╪══════════════════╪═══════╪═══════╡
│ synthetic ┆ AutoTheta        ┆ 0.755 ┆ 3.485 │
│ synthetic ┆ Chronos-2        ┆ 0.808 ┆ 0.796 │
│ synthetic ┆ TiRex-2          ┆ 0.878 ┆ 0.873 │
│ synthetic ┆ Timestral (123M) ┆ 0.788 ┆ 0.781 │
│ chaotic   ┆ AutoTheta        ┆ 5.718 ┆ 5.136 │
│ chaotic   ┆ Chronos-2        ┆ 3.658 ┆ 3.088 │
│ chaotic   ┆ TiRex-2          ┆ 3.583 ┆ 2.907 │
│ chaotic   ┆ Timestral (123M) ┆ 3.401 ┆ 2.751 │
│ real      ┆ AutoTheta        ┆ 2.428 ┆ 2.685 │
│ real      ┆ Chronos-2        ┆ 1.580 ┆ 1.352 │
│ real      ┆ TiRex-2          ┆ 1.679 ┆ 1.419 │
│ real      ┆ Timestral (123M) ┆ 1.769 ┆ 1.438 │
└───────────┴──────────────────┴───────┴───────┘


In [116]:
dataset_to_plot = "synthetic"  # choose from "synthetic", "chaotic", "real"
sample_to_plot = 5

forecasts = dict(zip(models.keys(), fc_results_spike_h[dataset_to_plot]))
plot_forecasts(
    data_samples_injected[dataset_to_plot][sample_to_plot],
    forecasts,
    index=sample_to_plot,
    exclude_from_y=[list(models)[0]],  # AutoTheta: its wide bands would squash the plot
)

/tmp/ipykernel_2215/2991431023.py:5: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  plot_forecasts(


#### Context only

In [117]:
inj = Injector(
    noise=False,
    spikes=True,
    shift=False,
    shift_covariate=False,
    spike_covariate=False,
    spike_region="context",
    spike_prob=0.04,
    seed=seed,
)

# inject anomalies into the samples
data_samples_injected = {}
for type_s, s in data_samples.items():
    print(f"Injecting anomalies into {type_s} samples..., number of samples: {len(s)}")
    data_samples_injected[type_s] = [inj(x, relative=type_s == "real") for x in s]

fc_results_spike_c = {}
fc_scores_spike_c = {}
for type_s, s in data_samples_injected.items():
    print(f"Forecasting {type_s} samples..., number of samples: {len(s)}")
    results, scores = evaluate_models_on_samples(models, s, clean_samples=data_samples[type_s])
    # save results and scores for a dataset
    fc_results_spike_c[type_s] = results
    fc_scores_spike_c[type_s] = scores

Injecting anomalies into synthetic samples..., number of samples: 100
Injecting anomalies into chaotic samples..., number of samples: 100
Injecting anomalies into real samples..., number of samples: 100
Forecasting synthetic samples..., number of samples: 100
Evaluating model: AutoTheta
Evaluating model: Chronos-2


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Evaluating model: TiRex-2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: Timestral (123M)


... (14 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: Timestral (123M)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [118]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=3):
    print(summarize_scores(fc_scores_spike_c))

shape: (12, 4)
┌───────────┬──────────────────┬───────┬───────┐
│ dataset   ┆ model            ┆ mase  ┆ sql   │
│ ---       ┆ ---              ┆ ---   ┆ ---   │
│ str       ┆ str              ┆ f64   ┆ f64   │
╞═══════════╪══════════════════╪═══════╪═══════╡
│ synthetic ┆ AutoTheta        ┆ 1.295 ┆ 1.877 │
│ synthetic ┆ Chronos-2        ┆ 0.131 ┆ 0.117 │
│ synthetic ┆ TiRex-2          ┆ 0.357 ┆ 0.323 │
│ synthetic ┆ Timestral (123M) ┆ 0.136 ┆ 0.162 │
│ chaotic   ┆ AutoTheta        ┆ 5.533 ┆ 6.021 │
│ chaotic   ┆ Chronos-2        ┆ 3.277 ┆ 2.668 │
│ chaotic   ┆ TiRex-2          ┆ 3.311 ┆ 2.640 │
│ chaotic   ┆ Timestral (123M) ┆ 3.120 ┆ 2.460 │
│ real      ┆ AutoTheta        ┆ 2.561 ┆ 2.257 │
│ real      ┆ Chronos-2        ┆ 1.483 ┆ 1.170 │
│ real      ┆ TiRex-2          ┆ 1.674 ┆ 1.283 │
│ real      ┆ Timestral (123M) ┆ 1.518 ┆ 1.177 │
└───────────┴──────────────────┴───────┴───────┘


In [119]:
dataset_to_plot = "synthetic"  # choose from "synthetic", "chaotic", "real"
sample_to_plot = 5

forecasts = dict(zip(models.keys(), fc_results_spike_c[dataset_to_plot]))
plot_forecasts(
    data_samples_injected[dataset_to_plot][sample_to_plot],
    forecasts,
    index=sample_to_plot,
    exclude_from_y=[list(models)[0]],  # AutoTheta: its wide bands would squash the plot
)

/tmp/ipykernel_2215/1422315983.py:5: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  plot_forecasts(


#### Both (period)

In [120]:
inj = Injector(
    noise=False,
    spikes=True,
    shift=False,
    shift_covariate=False,
    spike_covariate=False,
    spike_region="both",
    spike_prob=0.04,
    spike_period=48,
    seed=seed,
)

# inject anomalies into the samples
data_samples_injected = {}
for type_s, s in data_samples.items():
    print(f"Injecting anomalies into {type_s} samples..., number of samples: {len(s)}")
    data_samples_injected[type_s] = [inj(x, relative=type_s == "real") for x in s]

fc_results_spike_bp = {}
fc_scores_spike_bp = {}
for type_s, s in data_samples_injected.items():
    print(f"Forecasting {type_s} samples..., number of samples: {len(s)}")
    results, scores = evaluate_models_on_samples(models, s, clean_samples=data_samples[type_s])
    # save results and scores for a dataset
    fc_results_spike_bp[type_s] = results
    fc_scores_spike_bp[type_s] = scores

Injecting anomalies into synthetic samples..., number of samples: 100
Injecting anomalies into chaotic samples..., number of samples: 100
Injecting anomalies into real samples..., number of samples: 100
Forecasting synthetic samples..., number of samples: 100
Evaluating model: AutoTheta


Evaluating model: Chronos-2


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Evaluating model: TiRex-2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

... (15 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: Timestral (123M)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [121]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=3):
    print(summarize_scores(fc_scores_spike_bp))

shape: (12, 4)
┌───────────┬──────────────────┬───────┬───────┐
│ dataset   ┆ model            ┆ mase  ┆ sql   │
│ ---       ┆ ---              ┆ ---   ┆ ---   │
│ str       ┆ str              ┆ f64   ┆ f64   │
╞═══════════╪══════════════════╪═══════╪═══════╡
│ synthetic ┆ AutoTheta        ┆ 0.762 ┆ 1.619 │
│ synthetic ┆ Chronos-2        ┆ 0.106 ┆ 0.086 │
│ synthetic ┆ TiRex-2          ┆ 0.467 ┆ 0.409 │
│ synthetic ┆ Timestral (123M) ┆ 0.277 ┆ 0.227 │
│ chaotic   ┆ AutoTheta        ┆ 5.506 ┆ 5.928 │
│ chaotic   ┆ Chronos-2        ┆ 3.335 ┆ 2.742 │
│ chaotic   ┆ TiRex-2          ┆ 3.523 ┆ 2.821 │
│ chaotic   ┆ Timestral (123M) ┆ 3.309 ┆ 2.630 │
│ real      ┆ AutoTheta        ┆ 2.342 ┆ 2.948 │
│ real      ┆ Chronos-2        ┆ 1.378 ┆ 1.096 │
│ real      ┆ TiRex-2          ┆ 1.669 ┆ 1.303 │
│ real      ┆ Timestral (123M) ┆ 1.568 ┆ 1.246 │
└───────────┴──────────────────┴───────┴───────┘


In [122]:
dataset_to_plot = "synthetic"  # choose from "synthetic", "chaotic", "real"
sample_to_plot = 5

forecasts = dict(zip(models.keys(), fc_results_spike_bp[dataset_to_plot]))
plot_forecasts(
    data_samples_injected[dataset_to_plot][sample_to_plot],
    forecasts,
    index=sample_to_plot,
    exclude_from_y=[list(models)[0]],  # AutoTheta: its wide bands would squash the plot
)

/tmp/ipykernel_2215/1268868358.py:5: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  plot_forecasts(


#### Both (random)

In [123]:
inj = Injector(
    noise=False,
    spikes=True,
    shift=False,
    shift_covariate=False,
    spike_covariate=False,
    spike_region="both",
    spike_prob=0.04,
    spike_period=None,
    seed=seed,
)

# inject anomalies into the samples
data_samples_injected = {}
for type_s, s in data_samples.items():
    print(f"Injecting anomalies into {type_s} samples..., number of samples: {len(s)}")
    data_samples_injected[type_s] = [inj(x, relative=type_s == "real") for x in s]

fc_results_spike_br = {}
fc_scores_spike_br = {}
for type_s, s in data_samples_injected.items():
    print(f"Forecasting {type_s} samples..., number of samples: {len(s)}")
    results, scores = evaluate_models_on_samples(models, s, clean_samples=data_samples[type_s])
    # save results and scores for a dataset
    fc_results_spike_br[type_s] = results
    fc_scores_spike_br[type_s] = scores

Injecting anomalies into synthetic samples..., number of samples: 100
Injecting anomalies into chaotic samples..., number of samples: 100
Injecting anomalies into real samples..., number of samples: 100
Forecasting synthetic samples..., number of samples: 100
Evaluating model: AutoTheta


Evaluating model: Chronos-2


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Evaluating model: TiRex-2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

... (15 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: Timestral (123M)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [124]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=3):
    print(summarize_scores(fc_scores_spike_br))

shape: (12, 4)
┌───────────┬──────────────────┬───────┬───────┐
│ dataset   ┆ model            ┆ mase  ┆ sql   │
│ ---       ┆ ---              ┆ ---   ┆ ---   │
│ str       ┆ str              ┆ f64   ┆ f64   │
╞═══════════╪══════════════════╪═══════╪═══════╡
│ synthetic ┆ AutoTheta        ┆ 1.825 ┆ 2.431 │
│ synthetic ┆ Chronos-2        ┆ 0.899 ┆ 0.883 │
│ synthetic ┆ TiRex-2          ┆ 1.107 ┆ 1.067 │
│ synthetic ┆ Timestral (123M) ┆ 0.904 ┆ 0.918 │
│ chaotic   ┆ AutoTheta        ┆ 5.562 ┆ 5.987 │
│ chaotic   ┆ Chronos-2        ┆ 3.755 ┆ 3.131 │
│ chaotic   ┆ TiRex-2          ┆ 3.855 ┆ 3.117 │
│ chaotic   ┆ Timestral (123M) ┆ 3.582 ┆ 2.914 │
│ real      ┆ AutoTheta        ┆ 2.663 ┆ 2.446 │
│ real      ┆ Chronos-2        ┆ 1.671 ┆ 1.400 │
│ real      ┆ TiRex-2          ┆ 1.770 ┆ 1.464 │
│ real      ┆ Timestral (123M) ┆ 1.794 ┆ 1.468 │
└───────────┴──────────────────┴───────┴───────┘


In [125]:
dataset_to_plot = "synthetic"  # choose from "synthetic", "chaotic", "real"
sample_to_plot = 5

forecasts = dict(zip(models.keys(), fc_results_spike_br[dataset_to_plot]))
plot_forecasts(
    data_samples_injected[dataset_to_plot][sample_to_plot],
    forecasts,
    index=sample_to_plot,
    exclude_from_y=[list(models)[0]],  # AutoTheta: its wide bands would squash the plot
)

/tmp/ipykernel_2215/2361099547.py:5: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  plot_forecasts(


#### Summary

In [126]:
summary = summarize_scores({
    "clean": fc_scores,
    "spike_h": fc_scores_spike_h,
    "spike_c": fc_scores_spike_c,
    "spike_bp": fc_scores_spike_bp,
    "spike_br": fc_scores_spike_br,
})
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=3, tbl_width_chars=1000):
    print(summary)
    print(delta_scores(summary))

shape: (12, 12)
┌───────────┬──────────────────┬────────────┬──────────────┬──────────────┬───────────────┬───────────────┬───────────┬─────────────┬─────────────┬──────────────┬──────────────┐
│ dataset   ┆ model            ┆ mase_clean ┆ mase_spike_h ┆ mase_spike_c ┆ mase_spike_bp ┆ mase_spike_br ┆ sql_clean ┆ sql_spike_h ┆ sql_spike_c ┆ sql_spike_bp ┆ sql_spike_br │
│ ---       ┆ ---              ┆ ---        ┆ ---          ┆ ---          ┆ ---           ┆ ---           ┆ ---       ┆ ---         ┆ ---         ┆ ---          ┆ ---          │
│ str       ┆ str              ┆ f64        ┆ f64          ┆ f64          ┆ f64           ┆ f64           ┆ f64       ┆ f64         ┆ f64         ┆ f64          ┆ f64          │
╞═══════════╪══════════════════╪════════════╪══════════════╪══════════════╪═══════════════╪═══════════════╪═══════════╪═════════════╪═════════════╪══════════════╪══════════════╡
│ synthetic ┆ AutoTheta        ┆ 0.000      ┆ 0.755        ┆ 1.295        ┆ 0.762         ┆ 1.

### With covariates

#### Horizon only

In [127]:
inj = Injector(
    noise=False,
    spikes=True,
    shift=False,
    shift_covariate=False,
    spike_covariate=True,
    spike_region="horizon",
    spike_prob=0.04,
    seed=seed,
)

# inject anomalies into the samples
data_samples_injected = {}
for type_s, s in data_samples.items():
    print(f"Injecting anomalies into {type_s} samples..., number of samples: {len(s)}")
    data_samples_injected[type_s] = [inj(x, relative=type_s == "real") for x in s]

fc_results_spike_h_c = {}
fc_scores_spike_h_c = {}
for type_s, s in data_samples_injected.items():
    print(f"Forecasting {type_s} samples..., number of samples: {len(s)}")
    results, scores = evaluate_models_on_samples(models, s, clean_samples=data_samples[type_s])
    # save results and scores for a dataset
    fc_results_spike_h_c[type_s] = results
    fc_scores_spike_h_c[type_s] = scores

Injecting anomalies into synthetic samples..., number of samples: 100
Injecting anomalies into chaotic samples..., number of samples: 100
Injecting anomalies into real samples..., number of samples: 100
Forecasting synthetic samples..., number of samples: 100
Evaluating model: AutoTheta


Evaluating model: Chronos-2


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Evaluating model: TiRex-2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

... (15 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: Timestral (123M)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [128]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=3):
    print(summarize_scores(fc_scores_spike_h_c))

shape: (12, 4)
┌───────────┬──────────────────┬───────┬───────┐
│ dataset   ┆ model            ┆ mase  ┆ sql   │
│ ---       ┆ ---              ┆ ---   ┆ ---   │
│ str       ┆ str              ┆ f64   ┆ f64   │
╞═══════════╪══════════════════╪═══════╪═══════╡
│ synthetic ┆ AutoTheta        ┆ 0.755 ┆ 3.485 │
│ synthetic ┆ Chronos-2        ┆ 0.840 ┆ 0.821 │
│ synthetic ┆ TiRex-2          ┆ 0.871 ┆ 0.870 │
│ synthetic ┆ Timestral (123M) ┆ 0.792 ┆ 0.783 │
│ chaotic   ┆ AutoTheta        ┆ 5.718 ┆ 5.136 │
│ chaotic   ┆ Chronos-2        ┆ 3.720 ┆ 3.078 │
│ chaotic   ┆ TiRex-2          ┆ 3.571 ┆ 2.886 │
│ chaotic   ┆ Timestral (123M) ┆ 3.477 ┆ 2.783 │
│ real      ┆ AutoTheta        ┆ 2.428 ┆ 2.685 │
│ real      ┆ Chronos-2        ┆ 1.742 ┆ 1.469 │
│ real      ┆ TiRex-2          ┆ 1.665 ┆ 1.406 │
│ real      ┆ Timestral (123M) ┆ 1.813 ┆ 1.485 │
└───────────┴──────────────────┴───────┴───────┘


In [129]:
dataset_to_plot = "synthetic"  # choose from "synthetic", "chaotic", "real"
sample_to_plot = 5

forecasts = dict(zip(models.keys(), fc_results_spike_h_c[dataset_to_plot]))
plot_forecasts(
    data_samples_injected[dataset_to_plot][sample_to_plot],
    forecasts,
    index=sample_to_plot,
    exclude_from_y=[list(models)[0]],  # AutoTheta: its wide bands would squash the plot
)

/tmp/ipykernel_2215/115649765.py:5: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  plot_forecasts(


#### Context only

In [130]:
inj = Injector(
    noise=False,
    spikes=True,
    shift=False,
    shift_covariate=False,
    spike_covariate=True,
    spike_region="context",
    spike_prob=0.04,
    seed=seed,
)

# inject anomalies into the samples
data_samples_injected = {}
for type_s, s in data_samples.items():
    print(f"Injecting anomalies into {type_s} samples..., number of samples: {len(s)}")
    data_samples_injected[type_s] = [inj(x, relative=type_s == "real") for x in s]

fc_results_spike_c_c = {}
fc_scores_spike_c_c = {}
for type_s, s in data_samples_injected.items():
    print(f"Forecasting {type_s} samples..., number of samples: {len(s)}")
    results, scores = evaluate_models_on_samples(models, s, clean_samples=data_samples[type_s])
    # save results and scores for a dataset
    fc_results_spike_c_c[type_s] = results
    fc_scores_spike_c_c[type_s] = scores

Injecting anomalies into synthetic samples..., number of samples: 100
Injecting anomalies into chaotic samples..., number of samples: 100
Injecting anomalies into real samples..., number of samples: 100
Forecasting synthetic samples..., number of samples: 100
Evaluating model: AutoTheta


Evaluating model: Chronos-2


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Evaluating model: TiRex-2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

... (15 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: Timestral (123M)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [131]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=3):
    print(summarize_scores(fc_scores_spike_c_c))

shape: (12, 4)
┌───────────┬──────────────────┬───────┬───────┐
│ dataset   ┆ model            ┆ mase  ┆ sql   │
│ ---       ┆ ---              ┆ ---   ┆ ---   │
│ str       ┆ str              ┆ f64   ┆ f64   │
╞═══════════╪══════════════════╪═══════╪═══════╡
│ synthetic ┆ AutoTheta        ┆ 1.295 ┆ 1.877 │
│ synthetic ┆ Chronos-2        ┆ 0.099 ┆ 0.078 │
│ synthetic ┆ TiRex-2          ┆ 0.335 ┆ 0.304 │
│ synthetic ┆ Timestral (123M) ┆ 0.143 ┆ 0.131 │
│ chaotic   ┆ AutoTheta        ┆ 5.533 ┆ 6.021 │
│ chaotic   ┆ Chronos-2        ┆ 3.278 ┆ 2.633 │
│ chaotic   ┆ TiRex-2          ┆ 3.384 ┆ 2.671 │
│ chaotic   ┆ Timestral (123M) ┆ 3.195 ┆ 2.521 │
│ real      ┆ AutoTheta        ┆ 2.561 ┆ 2.257 │
│ real      ┆ Chronos-2        ┆ 1.626 ┆ 1.283 │
│ real      ┆ TiRex-2          ┆ 1.502 ┆ 1.161 │
│ real      ┆ Timestral (123M) ┆ 1.603 ┆ 1.206 │
└───────────┴──────────────────┴───────┴───────┘


In [132]:
dataset_to_plot = "synthetic"  # choose from "synthetic", "chaotic", "real"
sample_to_plot = 5

forecasts = dict(zip(models.keys(), fc_results_spike_c_c[dataset_to_plot]))
plot_forecasts(
    data_samples_injected[dataset_to_plot][sample_to_plot],
    forecasts,
    index=sample_to_plot,
    exclude_from_y=[list(models)[0]],  # AutoTheta: its wide bands would squash the plot
)

/tmp/ipykernel_2215/179687162.py:5: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  plot_forecasts(


#### Both (period)

In [133]:
inj = Injector(
    noise=False,
    spikes=True,
    shift=False,
    shift_covariate=False,
    spike_covariate=True,
    spike_region="both",
    spike_prob=0.04,
    spike_period=48,
    seed=seed,
)

# inject anomalies into the samples
data_samples_injected = {}
for type_s, s in data_samples.items():
    print(f"Injecting anomalies into {type_s} samples..., number of samples: {len(s)}")
    data_samples_injected[type_s] = [inj(x, relative=type_s == "real") for x in s]

fc_results_spike_bp_c = {}
fc_scores_spike_bp_c = {}
for type_s, s in data_samples_injected.items():
    print(f"Forecasting {type_s} samples..., number of samples: {len(s)}")
    results, scores = evaluate_models_on_samples(models, s, clean_samples=data_samples[type_s])
    # save results and scores for a dataset
    fc_results_spike_bp_c[type_s] = results
    fc_scores_spike_bp_c[type_s] = scores

Injecting anomalies into synthetic samples..., number of samples: 100
Injecting anomalies into chaotic samples..., number of samples: 100
Injecting anomalies into real samples..., number of samples: 100
Forecasting synthetic samples..., number of samples: 100
Evaluating model: AutoTheta


Evaluating model: Chronos-2


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Evaluating model: TiRex-2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

... (15 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: Timestral (123M)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [134]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=3):
    print(summarize_scores(fc_scores_spike_bp_c))

shape: (12, 4)
┌───────────┬──────────────────┬───────┬───────┐
│ dataset   ┆ model            ┆ mase  ┆ sql   │
│ ---       ┆ ---              ┆ ---   ┆ ---   │
│ str       ┆ str              ┆ f64   ┆ f64   │
╞═══════════╪══════════════════╪═══════╪═══════╡
│ synthetic ┆ AutoTheta        ┆ 0.762 ┆ 1.619 │
│ synthetic ┆ Chronos-2        ┆ 0.109 ┆ 0.088 │
│ synthetic ┆ TiRex-2          ┆ 0.358 ┆ 0.299 │
│ synthetic ┆ Timestral (123M) ┆ 0.224 ┆ 0.176 │
│ chaotic   ┆ AutoTheta        ┆ 5.506 ┆ 5.928 │
│ chaotic   ┆ Chronos-2        ┆ 3.232 ┆ 2.597 │
│ chaotic   ┆ TiRex-2          ┆ 3.340 ┆ 2.672 │
│ chaotic   ┆ Timestral (123M) ┆ 3.185 ┆ 2.513 │
│ real      ┆ AutoTheta        ┆ 2.342 ┆ 2.948 │
│ real      ┆ Chronos-2        ┆ 1.435 ┆ 1.154 │
│ real      ┆ TiRex-2          ┆ 1.455 ┆ 1.128 │
│ real      ┆ Timestral (123M) ┆ 1.492 ┆ 1.152 │
└───────────┴──────────────────┴───────┴───────┘


In [135]:
dataset_to_plot = "synthetic"  # choose from "synthetic", "chaotic", "real"
sample_to_plot = 5

forecasts = dict(zip(models.keys(), fc_results_spike_bp_c[dataset_to_plot]))
plot_forecasts(
    data_samples_injected[dataset_to_plot][sample_to_plot],
    forecasts,
    index=sample_to_plot,
    exclude_from_y=[list(models)[0]],  # AutoTheta: its wide bands would squash the plot
)

/tmp/ipykernel_2215/436871585.py:5: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  plot_forecasts(


#### Both (random)

In [136]:
inj = Injector(
    noise=False,
    spikes=True,
    shift=False,
    shift_covariate=False,
    spike_covariate=True,
    spike_region="both",
    spike_prob=0.04,
    spike_period=None,
    seed=seed,
)

# inject anomalies into the samples
data_samples_injected = {}
for type_s, s in data_samples.items():
    print(f"Injecting anomalies into {type_s} samples..., number of samples: {len(s)}")
    data_samples_injected[type_s] = [inj(x, relative=type_s == "real") for x in s]

fc_results_spike_br_c = {}
fc_scores_spike_br_c = {}
for type_s, s in data_samples_injected.items():
    print(f"Forecasting {type_s} samples..., number of samples: {len(s)}")
    results, scores = evaluate_models_on_samples(models, s, clean_samples=data_samples[type_s])
    # save results and scores for a dataset
    fc_results_spike_br_c[type_s] = results
    fc_scores_spike_br_c[type_s] = scores

Injecting anomalies into synthetic samples..., number of samples: 100
Injecting anomalies into chaotic samples..., number of samples: 100
Injecting anomalies into real samples..., number of samples: 100
Forecasting synthetic samples..., number of samples: 100
Evaluating model: AutoTheta


Evaluating model: Chronos-2


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Evaluating model: TiRex-2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

... (15 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: Timestral (123M)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [137]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=3):
    print(summarize_scores(fc_scores_spike_br_c))

shape: (12, 4)
┌───────────┬──────────────────┬───────┬───────┐
│ dataset   ┆ model            ┆ mase  ┆ sql   │
│ ---       ┆ ---              ┆ ---   ┆ ---   │
│ str       ┆ str              ┆ f64   ┆ f64   │
╞═══════════╪══════════════════╪═══════╪═══════╡
│ synthetic ┆ AutoTheta        ┆ 1.825 ┆ 2.431 │
│ synthetic ┆ Chronos-2        ┆ 0.462 ┆ 0.383 │
│ synthetic ┆ TiRex-2          ┆ 0.742 ┆ 0.634 │
│ synthetic ┆ Timestral (123M) ┆ 0.656 ┆ 0.538 │
│ chaotic   ┆ AutoTheta        ┆ 5.562 ┆ 5.987 │
│ chaotic   ┆ Chronos-2        ┆ 3.478 ┆ 2.743 │
│ chaotic   ┆ TiRex-2          ┆ 3.560 ┆ 2.813 │
│ chaotic   ┆ Timestral (123M) ┆ 3.302 ┆ 2.634 │
│ real      ┆ AutoTheta        ┆ 2.663 ┆ 2.446 │
│ real      ┆ Chronos-2        ┆ 1.495 ┆ 1.201 │
│ real      ┆ TiRex-2          ┆ 1.538 ┆ 1.212 │
│ real      ┆ Timestral (123M) ┆ 1.729 ┆ 1.354 │
└───────────┴──────────────────┴───────┴───────┘


In [138]:
dataset_to_plot = "synthetic"  # choose from "synthetic", "chaotic", "real"
sample_to_plot = 5

forecasts = dict(zip(models.keys(), fc_results_spike_br_c[dataset_to_plot]))
plot_forecasts(
    data_samples_injected[dataset_to_plot][sample_to_plot],
    forecasts,
    index=sample_to_plot,
    exclude_from_y=[list(models)[0]],  # AutoTheta: its wide bands would squash the plot
)

/tmp/ipykernel_2215/2868376069.py:5: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  plot_forecasts(


#### Summary

In [139]:
summary = summarize_scores({
    "clean": fc_scores,
    "spike_h_c": fc_scores_spike_h_c,
    "spike_c_c": fc_scores_spike_c_c,
    "spike_bp_c": fc_scores_spike_bp_c,
    "spike_br_c": fc_scores_spike_br_c,
})
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=3, tbl_width_chars=1000):
    print(summary)
    print(delta_scores(summary))

shape: (12, 12)
┌───────────┬──────────────────┬────────────┬────────────────┬────────────────┬─────────────────┬─────────────────┬───────────┬───────────────┬───────────────┬────────────────┬────────────────┐
│ dataset   ┆ model            ┆ mase_clean ┆ mase_spike_h_c ┆ mase_spike_c_c ┆ mase_spike_bp_c ┆ mase_spike_br_c ┆ sql_clean ┆ sql_spike_h_c ┆ sql_spike_c_c ┆ sql_spike_bp_c ┆ sql_spike_br_c │
│ ---       ┆ ---              ┆ ---        ┆ ---            ┆ ---            ┆ ---             ┆ ---             ┆ ---       ┆ ---           ┆ ---           ┆ ---            ┆ ---            │
│ str       ┆ str              ┆ f64        ┆ f64            ┆ f64            ┆ f64             ┆ f64             ┆ f64       ┆ f64           ┆ f64           ┆ f64            ┆ f64            │
╞═══════════╪══════════════════╪════════════╪════════════════╪════════════════╪═════════════════╪═════════════════╪═══════════╪═══════════════╪═══════════════╪════════════════╪════════════════╡
│ synthetic ┆ 

## Shift

### Without covariates

#### Horizon only

In [140]:
inj = Injector(
    noise=False,
    spikes=False,
    shift=True,
    shift_covariate=False,
    spike_covariate=False,
    shift_region="horizon",
    shift_position_range=(0.1, 0.6),
    seed=seed,
)

# inject anomalies into the samples
data_samples_injected = {}
for type_s, s in data_samples.items():
    print(f"Injecting anomalies into {type_s} samples..., number of samples: {len(s)}")
    data_samples_injected[type_s] = [inj(x, relative=type_s == "real") for x in s]

fc_results_shift_h = {}
fc_scores_shift_h = {}
for type_s, s in data_samples_injected.items():
    print(f"Forecasting {type_s} samples..., number of samples: {len(s)}")
    results, scores = evaluate_models_on_samples(models, s, clean_samples=data_samples[type_s])
    # save results and scores for a dataset
    fc_results_shift_h[type_s] = results
    fc_scores_shift_h[type_s] = scores

Injecting anomalies into synthetic samples..., number of samples: 100
Injecting anomalies into chaotic samples..., number of samples: 100
Injecting anomalies into real samples..., number of samples: 100
Forecasting synthetic samples..., number of samples: 100
Evaluating model: AutoTheta


Evaluating model: Chronos-2


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Evaluating model: TiRex-2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

... (15 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: Timestral (123M)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [141]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=3):
    print(summarize_scores(fc_scores_shift_h))

shape: (12, 4)
┌───────────┬──────────────────┬───────┬───────┐
│ dataset   ┆ model            ┆ mase  ┆ sql   │
│ ---       ┆ ---              ┆ ---   ┆ ---   │
│ str       ┆ str              ┆ f64   ┆ f64   │
╞═══════════╪══════════════════╪═══════╪═══════╡
│ synthetic ┆ AutoTheta        ┆ 3.264 ┆ 4.142 │
│ synthetic ┆ Chronos-2        ┆ 3.287 ┆ 3.256 │
│ synthetic ┆ TiRex-2          ┆ 3.312 ┆ 3.216 │
│ synthetic ┆ Timestral (123M) ┆ 3.277 ┆ 3.258 │
│ chaotic   ┆ AutoTheta        ┆ 6.923 ┆ 5.825 │
│ chaotic   ┆ Chronos-2        ┆ 5.158 ┆ 4.502 │
│ chaotic   ┆ TiRex-2          ┆ 5.154 ┆ 4.315 │
│ chaotic   ┆ Timestral (123M) ┆ 5.009 ┆ 4.189 │
│ real      ┆ AutoTheta        ┆ 3.498 ┆ 3.495 │
│ real      ┆ Chronos-2        ┆ 3.354 ┆ 3.064 │
│ real      ┆ TiRex-2          ┆ 3.398 ┆ 3.047 │
│ real      ┆ Timestral (123M) ┆ 3.496 ┆ 3.099 │
└───────────┴──────────────────┴───────┴───────┘


In [142]:
dataset_to_plot = "synthetic"  # choose from "synthetic", "chaotic", "real"
sample_to_plot = 5

forecasts = dict(zip(models.keys(), fc_results_shift_h[dataset_to_plot]))
plot_forecasts(
    data_samples_injected[dataset_to_plot][sample_to_plot],
    forecasts,
    index=sample_to_plot,
    exclude_from_y=[list(models)[0]],  # AutoTheta: its wide bands would squash the plot
)

/tmp/ipykernel_2215/3788883230.py:5: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  plot_forecasts(


#### Context only

In [143]:
inj = Injector(
    noise=False,
    spikes=False,
    shift=False,
    shift_covariate=False,
    spike_covariate=False,
    shift_region="context",
    shift_position_range=(0.2, 0.9),
    seed=seed,
)

ep_rng = np.random.default_rng(seed)
def context_episode(x, start_range=(0.3, 0.8), duration_range=(0.1, 0.25), min_gap=0.1):
    """One shift episode inside the context that ends at least min_gap * T before the forecast start."""
    T, H = len(x.context), len(x.target)
    duration = max(1, int(ep_rng.uniform(*duration_range) * T))
    lo = max(1, round(start_range[0] * T))
    hi = min(round(start_range[1] * T), T - duration - round(min_gap * T))  # latest start that still ends early enough
    start = int(ep_rng.integers(lo, max(hi, lo + 1)))
    end = min(start + duration, T - 1)
    return [(start / (T + H), end / (T + H))]

# inject anomalies into the samples
data_samples_injected = {}
for type_s, s in data_samples.items():
    print(f"Injecting anomalies into {type_s} samples..., number of samples: {len(s)}")
    data_samples_injected[type_s] = [inj(x, shift_episodes=context_episode(x), relative=type_s == "real") for x in s]

fc_results_shift_c = {}
fc_scores_shift_c = {}
for type_s, s in data_samples_injected.items():
    print(f"Forecasting {type_s} samples..., number of samples: {len(s)}")
    results, scores = evaluate_models_on_samples(models, s, clean_samples=data_samples[type_s])
    # save results and scores for a dataset
    fc_results_shift_c[type_s] = results
    fc_scores_shift_c[type_s] = scores

Injecting anomalies into synthetic samples..., number of samples: 100
Injecting anomalies into chaotic samples..., number of samples: 100
Injecting anomalies into real samples..., number of samples: 100
Forecasting synthetic samples..., number of samples: 100
Evaluating model: AutoTheta


Evaluating model: Chronos-2


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Evaluating model: TiRex-2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

... (15 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: Timestral (123M)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [144]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=3):
    print(summarize_scores(fc_scores_shift_c))

shape: (12, 4)
┌───────────┬──────────────────┬───────┬───────┐
│ dataset   ┆ model            ┆ mase  ┆ sql   │
│ ---       ┆ ---              ┆ ---   ┆ ---   │
│ str       ┆ str              ┆ f64   ┆ f64   │
╞═══════════╪══════════════════╪═══════╪═══════╡
│ synthetic ┆ AutoTheta        ┆ 0.571 ┆ 4.701 │
│ synthetic ┆ Chronos-2        ┆ 0.168 ┆ 0.168 │
│ synthetic ┆ TiRex-2          ┆ 0.230 ┆ 0.245 │
│ synthetic ┆ Timestral (123M) ┆ 0.821 ┆ 0.709 │
│ chaotic   ┆ AutoTheta        ┆ 5.322 ┆ 5.040 │
│ chaotic   ┆ Chronos-2        ┆ 3.258 ┆ 2.650 │
│ chaotic   ┆ TiRex-2          ┆ 3.225 ┆ 2.587 │
│ chaotic   ┆ Timestral (123M) ┆ 3.105 ┆ 2.465 │
│ real      ┆ AutoTheta        ┆ 2.091 ┆ 2.656 │
│ real      ┆ Chronos-2        ┆ 1.326 ┆ 1.036 │
│ real      ┆ TiRex-2          ┆ 1.500 ┆ 1.183 │
│ real      ┆ Timestral (123M) ┆ 1.415 ┆ 1.134 │
└───────────┴──────────────────┴───────┴───────┘


In [145]:
dataset_to_plot = "synthetic"  # choose from "synthetic", "chaotic", "real"
sample_to_plot = 5

forecasts = dict(zip(models.keys(), fc_results_shift_c[dataset_to_plot]))
plot_forecasts(
    data_samples_injected[dataset_to_plot][sample_to_plot],
    forecasts,
    index=sample_to_plot,
    exclude_from_y=[list(models)[0]],  # AutoTheta: its wide bands would squash the plot
)

/tmp/ipykernel_2215/4199610421.py:5: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  plot_forecasts(


#### Both

In [146]:
inj = Injector(
    noise=False,
    spikes=False,
    shift=True,
    shift_covariate=False,
    spike_covariate=False,
    shift_region="both",
    shift_position_range=(0.1, 0.9),
    seed=seed,
)

# inject anomalies into the samples
data_samples_injected = {}
for type_s, s in data_samples.items():
    print(f"Injecting anomalies into {type_s} samples..., number of samples: {len(s)}")
    data_samples_injected[type_s] = [inj(x, relative=type_s == "real") for x in s]

fc_results_shift_b = {}
fc_scores_shift_b = {}
for type_s, s in data_samples_injected.items():
    print(f"Forecasting {type_s} samples..., number of samples: {len(s)}")
    results, scores = evaluate_models_on_samples(models, s, clean_samples=data_samples[type_s])
    # save results and scores for a dataset
    fc_results_shift_b[type_s] = results
    fc_scores_shift_b[type_s] = scores

Injecting anomalies into synthetic samples..., number of samples: 100
Injecting anomalies into chaotic samples..., number of samples: 100
Injecting anomalies into real samples..., number of samples: 100
Forecasting synthetic samples..., number of samples: 100
Evaluating model: AutoTheta


Evaluating model: Chronos-2


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Evaluating model: TiRex-2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

... (15 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: Timestral (123M)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [147]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=3):
    print(summarize_scores(fc_scores_shift_b))

shape: (12, 4)
┌───────────┬──────────────────┬───────┬───────┐
│ dataset   ┆ model            ┆ mase  ┆ sql   │
│ ---       ┆ ---              ┆ ---   ┆ ---   │
│ str       ┆ str              ┆ f64   ┆ f64   │
╞═══════════╪══════════════════╪═══════╪═══════╡
│ synthetic ┆ AutoTheta        ┆ 3.277 ┆ 5.812 │
│ synthetic ┆ Chronos-2        ┆ 2.231 ┆ 1.869 │
│ synthetic ┆ TiRex-2          ┆ 2.893 ┆ 2.267 │
│ synthetic ┆ Timestral (123M) ┆ 0.704 ┆ 0.597 │
│ chaotic   ┆ AutoTheta        ┆ 7.866 ┆ 6.886 │
│ chaotic   ┆ Chronos-2        ┆ 6.355 ┆ 5.134 │
│ chaotic   ┆ TiRex-2          ┆ 6.008 ┆ 4.748 │
│ chaotic   ┆ Timestral (123M) ┆ 5.679 ┆ 4.517 │
│ real      ┆ AutoTheta        ┆ 3.462 ┆ 3.344 │
│ real      ┆ Chronos-2        ┆ 3.165 ┆ 2.550 │
│ real      ┆ TiRex-2          ┆ 2.875 ┆ 2.376 │
│ real      ┆ Timestral (123M) ┆ 2.861 ┆ 2.335 │
└───────────┴──────────────────┴───────┴───────┘


In [148]:
dataset_to_plot = "synthetic"  # choose from "synthetic", "chaotic", "real"
sample_to_plot = 5

forecasts = dict(zip(models.keys(), fc_results_shift_b[dataset_to_plot]))
plot_forecasts(
    data_samples_injected[dataset_to_plot][sample_to_plot],
    forecasts,
    index=sample_to_plot,
    exclude_from_y=[list(models)[0]],  # AutoTheta: its wide bands would squash the plot
)

/tmp/ipykernel_2215/2112557903.py:5: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  plot_forecasts(


#### Recurrent

In [149]:
inj = Injector(
    noise=False,
    spikes=False,
    shift=True,
    shift_covariate=False,
    spike_covariate=False,
    shift_region="any",
    shift_position_range=(0.1, 0.9),
    shift_recurring_prob=1.0,
    shift_duration_range=(0.1, 0.35),    
    seed=seed,
)

# inject anomalies into the samples
data_samples_injected = {}
for type_s, s in data_samples.items():
    print(f"Injecting anomalies into {type_s} samples..., number of samples: {len(s)}")
    data_samples_injected[type_s] = [inj(x, relative=type_s == "real") for x in s]

fc_results_shift_r = {}
fc_scores_shift_r = {}
for type_s, s in data_samples_injected.items():
    print(f"Forecasting {type_s} samples..., number of samples: {len(s)}")
    results, scores = evaluate_models_on_samples(models, s, clean_samples=data_samples[type_s])
    # save results and scores for a dataset
    fc_results_shift_r[type_s] = results
    fc_scores_shift_r[type_s] = scores

Injecting anomalies into synthetic samples..., number of samples: 100
Injecting anomalies into chaotic samples..., number of samples: 100
Injecting anomalies into real samples..., number of samples: 100
Forecasting synthetic samples..., number of samples: 100
Evaluating model: AutoTheta


Evaluating model: Chronos-2


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Evaluating model: TiRex-2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

... (15 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: Timestral (123M)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [150]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=3):
    print(summarize_scores(fc_scores_shift_r))

shape: (12, 4)
┌───────────┬──────────────────┬───────┬───────┐
│ dataset   ┆ model            ┆ mase  ┆ sql   │
│ ---       ┆ ---              ┆ ---   ┆ ---   │
│ str       ┆ str              ┆ f64   ┆ f64   │
╞═══════════╪══════════════════╪═══════╪═══════╡
│ synthetic ┆ AutoTheta        ┆ 1.657 ┆ 4.783 │
│ synthetic ┆ Chronos-2        ┆ 1.418 ┆ 1.240 │
│ synthetic ┆ TiRex-2          ┆ 1.472 ┆ 1.359 │
│ synthetic ┆ Timestral (123M) ┆ 1.654 ┆ 1.406 │
│ chaotic   ┆ AutoTheta        ┆ 5.863 ┆ 5.374 │
│ chaotic   ┆ Chronos-2        ┆ 3.974 ┆ 3.257 │
│ chaotic   ┆ TiRex-2          ┆ 3.890 ┆ 3.093 │
│ chaotic   ┆ Timestral (123M) ┆ 3.723 ┆ 2.943 │
│ real      ┆ AutoTheta        ┆ 2.811 ┆ 3.074 │
│ real      ┆ Chronos-2        ┆ 2.170 ┆ 1.778 │
│ real      ┆ TiRex-2          ┆ 2.265 ┆ 1.864 │
│ real      ┆ Timestral (123M) ┆ 2.122 ┆ 1.722 │
└───────────┴──────────────────┴───────┴───────┘


In [151]:
dataset_to_plot = "synthetic"  # choose from "synthetic", "chaotic", "real"
sample_to_plot = 6

forecasts = dict(zip(models.keys(), fc_results_shift_r[dataset_to_plot]))
plot_forecasts(
    data_samples_injected[dataset_to_plot][sample_to_plot],
    forecasts,
    index=sample_to_plot,
    exclude_from_y=[list(models)[0]],  # AutoTheta: its wide bands would squash the plot
)

/tmp/ipykernel_2215/168173860.py:5: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  plot_forecasts(


#### Summary

In [152]:
summary = summarize_scores({
    "clean": fc_scores,
    "shift_h": fc_scores_shift_h,
    "shift_c": fc_scores_shift_c,
    "shift_b": fc_scores_shift_b,
    "shift_r": fc_scores_shift_r,
})
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=3, tbl_width_chars=1000):
    print(summary)
    print(delta_scores(summary))

shape: (12, 12)
┌───────────┬──────────────────┬────────────┬──────────────┬──────────────┬──────────────┬──────────────┬───────────┬─────────────┬─────────────┬─────────────┬─────────────┐
│ dataset   ┆ model            ┆ mase_clean ┆ mase_shift_h ┆ mase_shift_c ┆ mase_shift_b ┆ mase_shift_r ┆ sql_clean ┆ sql_shift_h ┆ sql_shift_c ┆ sql_shift_b ┆ sql_shift_r │
│ ---       ┆ ---              ┆ ---        ┆ ---          ┆ ---          ┆ ---          ┆ ---          ┆ ---       ┆ ---         ┆ ---         ┆ ---         ┆ ---         │
│ str       ┆ str              ┆ f64        ┆ f64          ┆ f64          ┆ f64          ┆ f64          ┆ f64       ┆ f64         ┆ f64         ┆ f64         ┆ f64         │
╞═══════════╪══════════════════╪════════════╪══════════════╪══════════════╪══════════════╪══════════════╪═══════════╪═════════════╪═════════════╪═════════════╪═════════════╡
│ synthetic ┆ AutoTheta        ┆ 0.000      ┆ 3.264        ┆ 0.571        ┆ 3.277        ┆ 1.657        ┆ 3.014   

### With covariates

#### Horizon only

In [153]:
inj = Injector(
    noise=False,
    spikes=False,
    shift=True,
    shift_covariate=True,
    spike_covariate=False,
    shift_region="horizon",
    shift_position_range=(0.1, 0.6),
    seed=seed,
)

# inject anomalies into the samples
data_samples_injected = {}
for type_s, s in data_samples.items():
    print(f"Injecting anomalies into {type_s} samples..., number of samples: {len(s)}")
    data_samples_injected[type_s] = [inj(x, relative=type_s == "real") for x in s]

fc_results_shift_h_c = {}
fc_scores_shift_h_c = {}
for type_s, s in data_samples_injected.items():
    print(f"Forecasting {type_s} samples..., number of samples: {len(s)}")
    results, scores = evaluate_models_on_samples(models, s, clean_samples=data_samples[type_s])
    # save results and scores for a dataset
    fc_results_shift_h_c[type_s] = results
    fc_scores_shift_h_c[type_s] = scores

Injecting anomalies into synthetic samples..., number of samples: 100
Injecting anomalies into chaotic samples..., number of samples: 100
Injecting anomalies into real samples..., number of samples: 100
Forecasting synthetic samples..., number of samples: 100
Evaluating model: AutoTheta


Evaluating model: Chronos-2


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Evaluating model: TiRex-2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

... (15 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: Timestral (123M)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [154]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=3):
    print(summarize_scores(fc_scores_shift_h_c))

shape: (12, 4)
┌───────────┬──────────────────┬───────┬───────┐
│ dataset   ┆ model            ┆ mase  ┆ sql   │
│ ---       ┆ ---              ┆ ---   ┆ ---   │
│ str       ┆ str              ┆ f64   ┆ f64   │
╞═══════════╪══════════════════╪═══════╪═══════╡
│ synthetic ┆ AutoTheta        ┆ 3.264 ┆ 4.142 │
│ synthetic ┆ Chronos-2        ┆ 3.291 ┆ 3.223 │
│ synthetic ┆ TiRex-2          ┆ 3.312 ┆ 3.215 │
│ synthetic ┆ Timestral (123M) ┆ 3.272 ┆ 3.089 │
│ chaotic   ┆ AutoTheta        ┆ 6.923 ┆ 5.825 │
│ chaotic   ┆ Chronos-2        ┆ 5.187 ┆ 4.304 │
│ chaotic   ┆ TiRex-2          ┆ 5.134 ┆ 4.271 │
│ chaotic   ┆ Timestral (123M) ┆ 5.420 ┆ 5.773 │
│ real      ┆ AutoTheta        ┆ 3.498 ┆ 3.495 │
│ real      ┆ Chronos-2        ┆ 3.571 ┆ 3.209 │
│ real      ┆ TiRex-2          ┆ 3.487 ┆ 3.109 │
│ real      ┆ Timestral (123M) ┆ 3.804 ┆ 3.270 │
└───────────┴──────────────────┴───────┴───────┘


In [155]:
dataset_to_plot = "synthetic"  # choose from "synthetic", "chaotic", "real"
sample_to_plot = 5

forecasts = dict(zip(models.keys(), fc_results_shift_h_c[dataset_to_plot]))
plot_forecasts(
    data_samples_injected[dataset_to_plot][sample_to_plot],
    forecasts,
    index=sample_to_plot,
    exclude_from_y=[list(models)[0]],  # AutoTheta: its wide bands would squash the plot
)

/tmp/ipykernel_2215/1355630934.py:5: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  plot_forecasts(


#### Context only

In [156]:
inj = Injector(
    noise=False,
    spikes=False,
    shift=False,
    shift_covariate=True,
    spike_covariate=False,
    shift_region="context",
    shift_position_range=(0.2, 0.9),
    seed=seed,
)

ep_rng = np.random.default_rng(seed)
def context_episode(x, start_range=(0.3, 0.8), duration_range=(0.1, 0.25), min_gap=0.1):
    """One shift episode inside the context that ends at least min_gap * T before the forecast start."""
    T, H = len(x.context), len(x.target)
    duration = max(1, int(ep_rng.uniform(*duration_range) * T))
    lo = max(1, round(start_range[0] * T))
    hi = min(round(start_range[1] * T), T - duration - round(min_gap * T))  # latest start that still ends early enough
    start = int(ep_rng.integers(lo, max(hi, lo + 1)))
    end = min(start + duration, T - 1)
    return [(start / (T + H), end / (T + H))]

# inject anomalies into the samples
data_samples_injected = {}
for type_s, s in data_samples.items():
    print(f"Injecting anomalies into {type_s} samples..., number of samples: {len(s)}")
    data_samples_injected[type_s] = [inj(x, shift_episodes=context_episode(x), relative=type_s == "real") for x in s]

fc_results_shift_c_c = {}
fc_scores_shift_c_c = {}
for type_s, s in data_samples_injected.items():
    print(f"Forecasting {type_s} samples..., number of samples: {len(s)}")
    results, scores = evaluate_models_on_samples(models, s, clean_samples=data_samples[type_s])
    # save results and scores for a dataset
    fc_results_shift_c_c[type_s] = results
    fc_scores_shift_c_c[type_s] = scores

Injecting anomalies into synthetic samples..., number of samples: 100
Injecting anomalies into chaotic samples..., number of samples: 100
Injecting anomalies into real samples..., number of samples: 100
Forecasting synthetic samples..., number of samples: 100
Evaluating model: AutoTheta
Evaluating model: Chronos-2


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Evaluating model: TiRex-2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: Timestral (123M)


... (14 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: Timestral (123M)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [157]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=3):
    print(summarize_scores(fc_scores_shift_c_c))

shape: (12, 4)
┌───────────┬──────────────────┬───────┬───────┐
│ dataset   ┆ model            ┆ mase  ┆ sql   │
│ ---       ┆ ---              ┆ ---   ┆ ---   │
│ str       ┆ str              ┆ f64   ┆ f64   │
╞═══════════╪══════════════════╪═══════╪═══════╡
│ synthetic ┆ AutoTheta        ┆ 0.571 ┆ 4.701 │
│ synthetic ┆ Chronos-2        ┆ 0.075 ┆ 0.062 │
│ synthetic ┆ TiRex-2          ┆ 0.201 ┆ 0.198 │
│ synthetic ┆ Timestral (123M) ┆ 0.215 ┆ 0.178 │
│ chaotic   ┆ AutoTheta        ┆ 5.322 ┆ 5.040 │
│ chaotic   ┆ Chronos-2        ┆ 3.250 ┆ 2.574 │
│ chaotic   ┆ TiRex-2          ┆ 3.179 ┆ 2.516 │
│ chaotic   ┆ Timestral (123M) ┆ 3.066 ┆ 2.398 │
│ real      ┆ AutoTheta        ┆ 2.091 ┆ 2.656 │
│ real      ┆ Chronos-2        ┆ 1.406 ┆ 1.134 │
│ real      ┆ TiRex-2          ┆ 1.494 ┆ 1.191 │
│ real      ┆ Timestral (123M) ┆ 1.435 ┆ 1.104 │
└───────────┴──────────────────┴───────┴───────┘


In [158]:
dataset_to_plot = "synthetic"  # choose from "synthetic", "chaotic", "real"
sample_to_plot = 5

forecasts = dict(zip(models.keys(), fc_results_shift_c_c[dataset_to_plot]))
plot_forecasts(
    data_samples_injected[dataset_to_plot][sample_to_plot],
    forecasts,
    index=sample_to_plot,
    exclude_from_y=[list(models)[0]],  # AutoTheta: its wide bands would squash the plot
)

/tmp/ipykernel_2215/3929110697.py:5: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  plot_forecasts(


#### Both

In [159]:
inj = Injector(
    noise=False,
    spikes=False,
    shift=True,
    shift_covariate=True,
    spike_covariate=False,
    shift_region="both",
    shift_position_range=(0.1, 0.9),
    seed=seed,
)

# inject anomalies into the samples
data_samples_injected = {}
for type_s, s in data_samples.items():
    print(f"Injecting anomalies into {type_s} samples..., number of samples: {len(s)}")
    data_samples_injected[type_s] = [inj(x, relative=type_s == "real") for x in s]

fc_results_shift_b_c = {}
fc_scores_shift_b_c = {}
for type_s, s in data_samples_injected.items():
    print(f"Forecasting {type_s} samples..., number of samples: {len(s)}")
    results, scores = evaluate_models_on_samples(models, s, clean_samples=data_samples[type_s])
    # save results and scores for a dataset
    fc_results_shift_b_c[type_s] = results
    fc_scores_shift_b_c[type_s] = scores

Injecting anomalies into synthetic samples..., number of samples: 100
Injecting anomalies into chaotic samples..., number of samples: 100
Injecting anomalies into real samples..., number of samples: 100
Forecasting synthetic samples..., number of samples: 100
Evaluating model: AutoTheta
Evaluating model: Chronos-2


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Evaluating model: TiRex-2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: Timestral (123M)


... (14 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: Timestral (123M)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [160]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=3):
    print(summarize_scores(fc_scores_shift_b_c))

shape: (12, 4)
┌───────────┬──────────────────┬───────┬───────┐
│ dataset   ┆ model            ┆ mase  ┆ sql   │
│ ---       ┆ ---              ┆ ---   ┆ ---   │
│ str       ┆ str              ┆ f64   ┆ f64   │
╞═══════════╪══════════════════╪═══════╪═══════╡
│ synthetic ┆ AutoTheta        ┆ 3.277 ┆ 5.812 │
│ synthetic ┆ Chronos-2        ┆ 1.637 ┆ 1.359 │
│ synthetic ┆ TiRex-2          ┆ 1.720 ┆ 1.341 │
│ synthetic ┆ Timestral (123M) ┆ 0.540 ┆ 0.429 │
│ chaotic   ┆ AutoTheta        ┆ 7.866 ┆ 6.886 │
│ chaotic   ┆ Chronos-2        ┆ 5.980 ┆ 4.886 │
│ chaotic   ┆ TiRex-2          ┆ 6.086 ┆ 4.785 │
│ chaotic   ┆ Timestral (123M) ┆ 5.839 ┆ 4.620 │
│ real      ┆ AutoTheta        ┆ 3.462 ┆ 3.344 │
│ real      ┆ Chronos-2        ┆ 2.764 ┆ 2.267 │
│ real      ┆ TiRex-2          ┆ 2.581 ┆ 2.126 │
│ real      ┆ Timestral (123M) ┆ 2.741 ┆ 2.201 │
└───────────┴──────────────────┴───────┴───────┘


In [161]:
dataset_to_plot = "synthetic"  # choose from "synthetic", "chaotic", "real"
sample_to_plot = 5

forecasts = dict(zip(models.keys(), fc_results_shift_b_c[dataset_to_plot]))
plot_forecasts(
    data_samples_injected[dataset_to_plot][sample_to_plot],
    forecasts,
    index=sample_to_plot,
    exclude_from_y=[list(models)[0]],  # AutoTheta: its wide bands would squash the plot
)

/tmp/ipykernel_2215/3281833608.py:5: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  plot_forecasts(


#### Recurrent

In [162]:
inj = Injector(
    noise=False,
    spikes=False,
    shift=True,
    shift_covariate=True,
    spike_covariate=False,
    shift_region="any",
    shift_position_range=(0.1, 0.9),
    shift_recurring_prob=1.0,
    shift_duration_range=(0.1, 0.35),    
    seed=seed,
)

# inject anomalies into the samples
data_samples_injected = {}
for type_s, s in data_samples.items():
    print(f"Injecting anomalies into {type_s} samples..., number of samples: {len(s)}")
    data_samples_injected[type_s] = [inj(x, relative=type_s == "real") for x in s]

fc_results_shift_r_c = {}
fc_scores_shift_r_c = {}
for type_s, s in data_samples_injected.items():
    print(f"Forecasting {type_s} samples..., number of samples: {len(s)}")
    results, scores = evaluate_models_on_samples(models, s, clean_samples=data_samples[type_s])
    # save results and scores for a dataset
    fc_results_shift_r_c[type_s] = results
    fc_scores_shift_r_c[type_s] = scores

Injecting anomalies into synthetic samples..., number of samples: 100
Injecting anomalies into chaotic samples..., number of samples: 100
Injecting anomalies into real samples..., number of samples: 100
Forecasting synthetic samples..., number of samples: 100
Evaluating model: AutoTheta


Evaluating model: Chronos-2


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Evaluating model: TiRex-2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

... (15 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: Timestral (123M)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [163]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=3):
    print(summarize_scores(fc_scores_shift_r_c))

shape: (12, 4)
┌───────────┬──────────────────┬───────┬───────┐
│ dataset   ┆ model            ┆ mase  ┆ sql   │
│ ---       ┆ ---              ┆ ---   ┆ ---   │
│ str       ┆ str              ┆ f64   ┆ f64   │
╞═══════════╪══════════════════╪═══════╪═══════╡
│ synthetic ┆ AutoTheta        ┆ 1.657 ┆ 4.783 │
│ synthetic ┆ Chronos-2        ┆ 0.459 ┆ 0.395 │
│ synthetic ┆ TiRex-2          ┆ 0.614 ┆ 0.549 │
│ synthetic ┆ Timestral (123M) ┆ 0.539 ┆ 0.448 │
│ chaotic   ┆ AutoTheta        ┆ 5.863 ┆ 5.374 │
│ chaotic   ┆ Chronos-2        ┆ 3.568 ┆ 2.877 │
│ chaotic   ┆ TiRex-2          ┆ 3.534 ┆ 2.828 │
│ chaotic   ┆ Timestral (123M) ┆ 3.266 ┆ 2.571 │
│ real      ┆ AutoTheta        ┆ 2.811 ┆ 3.074 │
│ real      ┆ Chronos-2        ┆ 1.639 ┆ 1.343 │
│ real      ┆ TiRex-2          ┆ 1.879 ┆ 1.506 │
│ real      ┆ Timestral (123M) ┆ 1.756 ┆ 1.420 │
└───────────┴──────────────────┴───────┴───────┘


In [164]:
dataset_to_plot = "synthetic"  # choose from "synthetic", "chaotic", "real"
sample_to_plot = 6

forecasts = dict(zip(models.keys(), fc_results_shift_r_c[dataset_to_plot]))
plot_forecasts(
    data_samples_injected[dataset_to_plot][sample_to_plot],
    forecasts,
    index=sample_to_plot,
    exclude_from_y=[list(models)[0]],  # AutoTheta: its wide bands would squash the plot
)

/tmp/ipykernel_2215/2938354514.py:5: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  plot_forecasts(


#### Summary

In [165]:
summary = summarize_scores({
    "clean": fc_scores,
    "shift_h_c": fc_scores_shift_h_c,
    "shift_c_c": fc_scores_shift_c_c,
    "shift_b_c": fc_scores_shift_b_c,
    "shift_r_c": fc_scores_shift_r_c,
})
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=3, tbl_width_chars=1000):
    print(summary)
    print(delta_scores(summary))

shape: (12, 12)
┌───────────┬──────────────────┬────────────┬────────────────┬────────────────┬────────────────┬────────────────┬───────────┬───────────────┬───────────────┬───────────────┬───────────────┐
│ dataset   ┆ model            ┆ mase_clean ┆ mase_shift_h_c ┆ mase_shift_c_c ┆ mase_shift_b_c ┆ mase_shift_r_c ┆ sql_clean ┆ sql_shift_h_c ┆ sql_shift_c_c ┆ sql_shift_b_c ┆ sql_shift_r_c │
│ ---       ┆ ---              ┆ ---        ┆ ---            ┆ ---            ┆ ---            ┆ ---            ┆ ---       ┆ ---           ┆ ---           ┆ ---           ┆ ---           │
│ str       ┆ str              ┆ f64        ┆ f64            ┆ f64            ┆ f64            ┆ f64            ┆ f64       ┆ f64           ┆ f64           ┆ f64           ┆ f64           │
╞═══════════╪══════════════════╪════════════╪════════════════╪════════════════╪════════════════╪════════════════╪═══════════╪═══════════════╪═══════════════╪═══════════════╪═══════════════╡
│ synthetic ┆ AutoTheta        ┆ 0